In [ ]:
import os
import pickle
import cv2
import face_recognition

# Chemins relatifs par rapport au dossier notebook/
DATASET_PATH = "../dataset"
MODEL_OUTPUT_DIR = "../models"
MODEL_OUTPUT_FILE = os.path.join(MODEL_OUTPUT_DIR, "face_encodings.pkl")

# Création automatique du dossier models s'il n'existe pas
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
print(f"Dossier de sortie prêt : {MODEL_OUTPUT_DIR}")

In [ ]:
def extraire_encodages(dataset_dir):
    """
    Parcourt les sous-dossiers de chaque étudiant et extrait les encodages 128D.
    """
    known_encodings = []
    known_names = []

    if not os.path.exists(dataset_dir):
        print(f"Erreur : Le dossier {dataset_dir} n'existe pas encore.")
        return known_encodings, known_names

    # Liste des dossiers des étudiants
    personnes = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))]
    print(f"Classes détectées ({len(personnes)}) : {personnes}")

    for nom in personnes:
        dossier_personne = os.path.join(dataset_dir, nom)
        images = os.listdir(dossier_personne)
        print(f"Traitement de {nom} : {len(images)} images trouvées...")

        count_visages = 0
        for image_name in images:
            image_path = os.path.join(dossier_personne, image_name)
            
            # Chargement de l'image
            image = cv2.imread(image_path)
            if image is None:
                continue

            # Conversion standard BGR (OpenCV) vers RGB (dlib/face_recognition)
            rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Localisation et encodage du visage
            boxes = face_recognition.face_locations(rgb_image, model="hog")
            encodings = face_recognition.face_encodings(rgb_image, boxes)

            # On conserve les encodages valides
            for encoding in encodings:
                known_encodings.append(encoding)
                known_names.append(nom)
                count_visages += 1

        print(f"  -> {count_visages} signatures encodées pour {nom}.")

    return known_encodings, known_names

In [ ]:
# 1. Extraction
encodings, names = extraire_encodages(DATASET_PATH)

# 2. Sauvegarde dans models/face_encodings.pkl
if len(encodings) > 0:
    data = {
        "encodings": encodings,
        "names": names
    }
    with open(MODEL_OUTPUT_FILE, "wb") as f:
        pickle.dump(data, f)
    print(f"[Succès] {len(encodings)} visages sauvegardés dans {MODEL_OUTPUT_FILE}")
else:
    print("[Attention] Aucun visage n'a été encodé. Vérifiez le contenu du dossier dataset.")